# 04 · Pandas 표 탐색과 선택

열 이름으로 데이터를 읽고, 행과 열을 선택하고, 같은 결과를 만드는 표현을 비교합니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## 표 만들기와 읽기

**코드 → 코드 개념**: DataFrame은 열 이름과 행 인덱스를 가진 2차원 표다.

**코드 사용법**: 이 노트북의 예제 표를 만들고 기본 구조를 점검한다.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "machine": ["A", "A", "B", "B", "B"],
    "cycle": [1, 2, 1, 2, 3],
    "temperature": [72, 83, 75, 80, 85],
    "vibration": [2.1, 3.5, 2.4, 2.9, 4.1],
})
print(df.head(), df.tail(2))
print(df.shape, df.columns.tolist(), df.dtypes)
df.info()
print(df.describe(include="all"))

**같은 결과를 얻는 방법과 선택 이유**

- 실제 CSV는 `pd.read_csv('file.csv')`로 읽는다. `head`는 행 예시, `info`는 자료형과 비결측 수, `describe`는 분포를 본다. 서로 대체 관계가 아니라 다른 질문에 답한다.
- `describe()`는 기본적으로 숫자 열만, `include='all'`은 문자 열도 요약한다.

## Series와 DataFrame, loc와 iloc

**코드 → 코드 개념**: 한 열 선택은 Series, 여러 열 선택은 DataFrame이다. `loc`는 라벨, `iloc`는 위치다.

**코드 사용법**: 각 표현의 결과 형태를 비교한다.

In [ ]:
print(type(df["temperature"]), type(df[["temperature"]]))
print(df.loc[df["machine"] == "A", ["cycle", "temperature"]])
print(df.iloc[:2, 1:3])
print(df["temperature"].iloc[0], df.loc[0, "temperature"])

**같은 결과를 얻는 방법과 선택 이유**

- 이름을 알고 있으면 `loc`가 열의 의미를 보여 준다. 위치로 처음 n행·n열을 살필 때 `iloc`가 편하다. 인덱스가 0,1,2가 아니면 `loc[0]`과 `iloc[0]`은 다른 행일 수 있다.
- 한 열이어도 2차원 입력이 필요한 모델에는 `df[['temperature']]`을 쓴다.

## 조건 필터링의 여러 표현

**코드 → 코드 개념**: 불리언 Series는 각 행을 남길지 결정한다.

**코드 사용법**: 75~85℃이고 설비 B인 행을 세 방식으로 찾는다.

In [ ]:
a = df.loc[(df["temperature"] >= 75) & (df["temperature"] <= 85) & (df["machine"] == "B")]
b = df.loc[df["temperature"].between(75, 85) & df["machine"].isin(["B"])]
c = df.query("75 <= temperature <= 85 and machine == 'B'")
print(a, b, c, sep="\n")
assert a.equals(b) and b.equals(c)

**같은 결과를 얻는 방법과 선택 이유**

- `between`은 닫힌 구간을 간단히 표현하고 `isin`은 여러 값 중 하나를 고를 때 좋다.
- `query`는 식이 길 때 읽기 쉽지만 공백·특수문자가 있는 열 이름은 백틱 처리해야 한다. 변수와 열 이름이 섞이면 `loc`가 더 분명하다.

## 정렬과 복사

**코드 → 코드 개념**: `sort_values`는 행 순서를 바꾼 새 표를 만든다. `.copy()`는 독립된 결과를 만든다.

**코드 사용법**: 필터링한 행에 경고 열을 추가한다.

In [ ]:
selected = df.loc[df["vibration"] >= 3].copy()
selected["warning"] = True
print(selected.sort_values(["machine", "vibration"], ascending=[True, False]))
print("원본 열:", df.columns.tolist())

**같은 결과를 얻는 방법과 선택 이유**

- 원본이 필요하면 새 변수에 정렬 결과를 담는다. `inplace=True`보다 결과를 할당하는 방식이 데이터 흐름을 따라가기 쉽다.
- 필터 결과에 값을 대입할 때 `.copy()`를 쓰거나 `df.loc[mask, 'warning'] = ...`로 원본에 직접 대입한다.

## 값 빈도와 구간화

**코드 → 코드 개념**: `value_counts`는 범주별 빈도를, `cut`은 연속값을 구간으로 바꾼다.

**코드 사용법**: 설비 빈도와 진동 등급을 확인한다.

In [ ]:
print(df["machine"].value_counts())
print(df["machine"].value_counts(normalize=True))
df["vibration_band"] = pd.cut(df["vibration"], bins=[0, 3, 4, float("inf")], labels=["normal", "watch", "danger"])
print(df[["vibration", "vibration_band"]])

**같은 결과를 얻는 방법과 선택 이유**

- `normalize=True`는 건수 대신 비율을 준다. 설비마다 측정 건수가 다르면 비율과 건수를 같이 본다.
- `pd.cut`은 미리 정한 물리적 기준에, `pd.qcut`은 데이터의 분위수로 비슷한 건수의 구간을 만들 때 쓴다. `cut` 경계의 포함 방향도 확인한다.

## 원본 학습 자료

[`1. lecture/02_Pandas/basic`](../1.%20lecture/02_Pandas/basic), [`4. Summary/07_Pandas`](../4.%20Summary/07_Pandas)